# Week 3 Pipeline: Product Compare Assistant

**Project:** RAG-based headphone comparison assistant. **Notebook:** `notebooks/week3_pipeline.ipynb`

Two data types are prepared, each with its own pathway, and one manifest links their outputs:

| Branch | Sources | Pathway |
|---|---|---|
| JSON | S01 product metadata, S02 customer reviews (Amazon Reviews 2023, headphones) | D (JSON/API export) |
| Documents | S03, S04, S05 (open-licence research papers, PDF) | A (PDF/TXT) |

**Run order (REQUIRED):** Cell 1 → Cell 2 → Cell 3 (download S01, S02) → Cell 4 (upload PDFs) → Cell 5 (verify) → Cell 6 → Cell 7 → Cell 8 (helpers) → Cell 9 (inspect) → Cells 10, 11, 12 (Part B pathways) → Cell 13 (manifest) → Cell 14 (verify manifest) → Cells 15, 16, 17 (Part C pipeline) → Cell 18 (repeatability) → Cell 19 (package for GitHub).

After a runtime restart, rerun from Cell 1. When the run is complete, use **File → Download .ipynb** so the saved outputs are kept as evidence.

## Setup

In [11]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [12]:
# Cell 2 — Project settings (Product Compare Assistant)
# Everything that identifies a source or a configuration is defined here once.

# Dataset revision pinned from the Week 3 inspection, so every run reads the same data.
HF_REPO = "McAuley-Lab/Amazon-Reviews-2023"
HF_REVISION = "2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e"
HEADPHONE_BRANCH = "Headphones & Earbuds"   # categories[2] in the Electronics metadata
REVIEW_SCAN_ROWS = 500_000                  # first rows of the review file that are scanned

S01_NAME = "s01_headphones_meta.jsonl"
S02_NAME = "s02_headphones_reviews.jsonl"
PDF_NAMES = {
    "S03": "s03_lee_2025_wireless_earbuds.pdf",
    "S04": "s04_gutierrez_parera_2016_headphone_quality.pdf",
    "S05": "s05_valimaki_reiss_2016_equalization.pdf",
}

# Part of the original download name -> canonical file name (used by Cell 4)
UPLOAD_RENAME = {
    "perceived_risk": PDF_NAMES["S03"],
    "applsci-06-00117": PDF_NAMES["S04"],
    "applsci-06-00129": PDF_NAMES["S05"],
}

# Source identity and rights, copied from docs/week3_data_inventory.md
SOURCE_META = {
    S01_NAME: {
        "source_id": "S01",
        "title": "Amazon Reviews 2023, Electronics item metadata, headphones extract",
        "version": f"{HF_REPO}, config raw_meta_Electronics, revision {HF_REVISION}",
        "access_rights": "Public download. No licence stated on the dataset card (checked 2/11/2026). Coursework use only; raw and full prepared data are not committed.",
        "limitations": "Licence not stated; 2023 snapshot; description present for about 57% of products; price for about 22%.",
    },
    S02_NAME: {
        "source_id": "S02",
        "title": "Amazon Reviews 2023, Electronics reviews, headphones extract",
        "version": f"{HF_REPO}, config raw_review_Electronics, revision {HF_REVISION}, first {REVIEW_SCAN_ROWS} rows scanned",
        "access_rights": "Same rights as S01. Review text written by Amazon customers; user_id is dropped from prepared data.",
        "limitations": "Licence not stated; only the first scanned rows are used, so the subset is not a random sample of all reviews.",
    },
    PDF_NAMES["S03"]: {
        "source_id": "S03",
        "title": "Perceived risk and fashion on the intention to adopt wireless earbuds in the United States",
        "doi": "10.2196/56887",
        "version": "JMIR Formative Research 9, e56887 (2025)",
        "licence": "CC BY 4.0",
        "access_rights": "Open access, CC BY 4.0 (article page footer, checked 4/11/2026). Reuse with credit.",
        "limitations": "Contains health-risk discussion; FR8 forbids health advice.",
    },
    PDF_NAMES["S04"]: {
        "source_id": "S04",
        "title": "Influence of the quality of consumer headphones in the perception of spatial audio",
        "doi": "10.3390/app6040117",
        "version": "Applied Sciences 6(4), 117 (2016)",
        "licence": "CC BY 4.0",
        "access_rights": "Open access, CC BY 4.0. Confirm the version printed in the PDF.",
        "limitations": "Small listening test with simulated headphones.",
    },
    PDF_NAMES["S05"]: {
        "source_id": "S05",
        "title": "All about audio equalization: Solutions and frontiers",
        "doi": "10.3390/app6050129",
        "version": "Applied Sciences 6(5), 129 (2016)",
        "licence": "CC BY 4.0",
        "access_rights": "Open access, CC BY 4.0. Confirm the version printed in the PDF.",
        "limitations": "Only Sections 1 and 6 are kept.",
    },
}
print("Settings loaded. Sources:", [m["source_id"] for m in SOURCE_META.values()])

Settings loaded. Sources: ['S01', 'S02', 'S03', 'S04', 'S05']


## Part A — Ingest the sources into `data/raw`

In [13]:
# Cell 3 — Download S01 and S02 from Hugging Face (repeatable, pinned revision)
# Skips the download when the extract already exists. Set FORCE_REBUILD = True to redo it.
!pip -q install "datasets<4.0" pypdf

import json
import itertools
from datasets import load_dataset

S01_PATH = RAW_DIR / S01_NAME
S02_PATH = RAW_DIR / S02_NAME
FORCE_REBUILD = False

def is_headphone_branch(categories):
    return bool(categories) and len(categories) >= 3 and categories[2] == HEADPHONE_BRANCH

# --- S01: product metadata, filtered to the headphone branch ---
if S01_PATH.is_file() and not FORCE_REBUILD:
    print("S01 extract already exists:", S01_PATH.name)
else:
    meta = load_dataset(HF_REPO, "raw_meta_Electronics", split="full",
                        trust_remote_code=True, revision=HF_REVISION)
    print("Electronics metadata rows:", len(meta))
    headphones = meta.filter(lambda ex: is_headphone_branch(ex.get("categories")), num_proc=4)
    with S01_PATH.open("w", encoding="utf-8") as out:
        for ex in headphones:
            out.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print("S01 headphone products written:", len(headphones))

# --- S02: reviews, streamed; keep reviews of the S01 products found in the first rows ---
if S02_PATH.is_file() and not FORCE_REBUILD:
    print("S02 extract already exists:", S02_PATH.name)
else:
    headphone_ids = set()
    with S01_PATH.open("r", encoding="utf-8") as handle:
        for line in handle:
            headphone_ids.add(json.loads(line)["parent_asin"])
    stream = load_dataset(HF_REPO, "raw_review_Electronics", split="full", streaming=True,
                          trust_remote_code=True, revision=HF_REVISION)
    scanned = kept = 0
    with S02_PATH.open("w", encoding="utf-8") as out:
        for rec in itertools.islice(stream, REVIEW_SCAN_ROWS):
            scanned += 1
            if rec["parent_asin"] in headphone_ids:
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                kept += 1
    print("S02 rows scanned:", scanned, "| headphone reviews written:", kept)

S01 extract already exists: s01_headphones_meta.jsonl
S02 extract already exists: s02_headphones_reviews.jsonl


In [14]:
# Cell 4 — Upload S03, S04, S05 (the three PDFs) and give them canonical names
# Skips the upload when all three canonical files are already in data/raw.
missing = [name for name in PDF_NAMES.values() if not (RAW_DIR / name).is_file()]

if not missing:
    print("All three PDFs are already in data/raw.")
else:
    from google.colab import files
    import re
    print("Select the three PDFs:", missing)
    uploaded = files.upload()          # files are saved in the current folder (/content)
    for original in uploaded:
        # Clean duplicate suffix (e.g. ' (1).pdf' -> '.pdf')
        clean_original = re.sub(r"\s*\(\d+\)\.pdf$", ".pdf", original.lower(), flags=re.IGNORECASE)

        # Check direct canonical match first
        matched = False
        for canon_name in PDF_NAMES.values():
            if clean_original == canon_name.lower():
                (Path.cwd() / original).replace(RAW_DIR / canon_name)
                print("Saved direct match:", original, "->", canon_name)
                matched = True
                break
        if not matched:
            # Try matching via UPLOAD_RENAME keys
            for key, canonical in UPLOAD_RENAME.items():
                if key in clean_original or canonical.lower() in clean_original:
                    (Path.cwd() / original).replace(RAW_DIR / canonical)
                    print("Saved pattern match:", original, "->", canonical)
                    break
    still_missing = [name for name in PDF_NAMES.values() if not (RAW_DIR / name).is_file()]
    assert not still_missing, f"Still missing: {still_missing}"

All three PDFs are already in data/raw.


In [15]:
# Cell 5 — Verify uploaded inputs (REQUIRED, from the lab)
from pathlib import Path

RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files_found = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files_found))
for f in files_found:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files_found, "Upload at least one relevant input file before proceeding."
expected = [S01_NAME, S02_NAME] + list(PDF_NAMES.values())
absent = [name for name in expected if not (RAW / name).is_file()]
assert not absent, f"Missing expected sources: {absent}"
print("Upload verified. All five project sources are present.")

Input count: 5
s01_headphones_meta.jsonl bytes: 191069513
s02_headphones_reviews.jsonl bytes: 25226541
s03_lee_2025_wireless_earbuds.pdf bytes: 1112260
s04_gutierrez_parera_2016_headphone_quality.pdf bytes: 1340749
s05_valimaki_reiss_2016_equalization.pdf bytes: 1928577
Upload verified. All five project sources are present.


In [16]:
# Cell 6 — REQUIRED: Source selection + lineage utilities (from the lab)
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing. Never take the first file by extension.
def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records,
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 6 completed successfully.")
print("Available functions: select_source(), source_hash(), save_manifest()")

Cell 6 completed successfully.
Available functions: select_source(), source_hash(), save_manifest()


In [17]:
# Cell 7 — REQUIRED: Initialise the common lineage registry (Cell 4A in the lab)
from pathlib import Path

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    # Source and output must belong to the project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # The same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {"source": source, "output": output, "processing": processing}
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


## Project helpers (used by both Part B and Part C)

In [18]:
# Cell 8 — Project helpers. The same functions run in Part B and in the Part C pipeline,
# so the configuration in Part C is guaranteed to match Part B.
import re
import html
import json
import hashlib
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from pypdf import PdfReader

EVIDENCE = {}   # filled by the Part B cells, written to a file by Cell 19

def iter_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)

def read_jsonl(path, limit=None):
    rows = []
    for row in iter_jsonl(path):
        rows.append(row)
        if limit and len(rows) >= limit:
            break
    return rows

def non_blank(value):
    return isinstance(value, str) and value.strip() not in ("", "None")

def shorten(value, max_str=160, max_items=3):
    if isinstance(value, str):
        return value if len(value) <= max_str else value[:max_str] + "...[cut]"
    if isinstance(value, list):
        out = [shorten(v, max_str, max_items) for v in value[:max_items]]
        if len(value) > max_items:
            out.append(f"...[{len(value) - max_items} more]")
        return out
    if isinstance(value, dict):
        return {k: shorten(v, max_str, max_items) for k, v in value.items()}
    return value

# ----------------------------- S01: product metadata -----------------------------
S01_REQUIRED_KEYS = ["parent_asin", "title"]

def meta_is_valid(rec, config):
    """Config A: required keys present and not blank.
    Config B: config A plus at least one non-blank features or description text."""
    if not isinstance(rec, dict) or not all(k in rec for k in S01_REQUIRED_KEYS):
        return False
    if not (non_blank(rec["parent_asin"]) and non_blank(rec["title"])):
        return False
    if config == "B":
        has_text = (any(non_blank(t) for t in (rec.get("features") or []))
                    or any(non_blank(t) for t in (rec.get("description") or [])))
        return has_text
    return True

def parse_details(value):
    """The details field is a JSON string. Return (dict or None, parsed_ok)."""
    if isinstance(value, dict):
        return value, True
    if non_blank(value):
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            return None, False
        return (parsed, True) if isinstance(parsed, dict) else (None, False)
    return None, True

def to_price(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def prepare_meta_record(rec, config="A"):
    categories = rec.get("categories") or []
    details, details_ok = parse_details(rec.get("details"))
    features = [t.strip() for t in (rec.get("features") or []) if non_blank(t)]
    description = [t.strip() for t in (rec.get("description") or []) if non_blank(t)]
    price = to_price(rec.get("price"))
    return {
        "record_id": f"meta-{rec['parent_asin']}",
        "source_id": "S01",
        "parent_asin": rec["parent_asin"],
        "title": rec["title"].strip(),
        "store": rec.get("store"),
        "categories": categories,
        "subtype": categories[3] if len(categories) > 3 else None,
        "average_rating": rec.get("average_rating"),
        "rating_number": rec.get("rating_number"),
        "price_2023_snapshot": price,
        "features": features,
        "description": description,
        "details": details,
        "details_parsed": details_ok,
        "has_features": bool(features),
        "has_description": bool(description),
        "has_price": price is not None,
    }

def compare_meta_configs(src):
    total = 0
    valid = {"A": 0, "B": 0}
    stores = {"A": set(), "B": set()}
    for rec in iter_jsonl(src):
        total += 1
        for cfg in ("A", "B"):
            if meta_is_valid(rec, cfg):
                valid[cfg] += 1
                if non_blank(rec.get("store")):
                    stores[cfg].add(rec["store"])
    return {
        "records_in": total,
        "A_valid": valid["A"], "B_valid": valid["B"],
        "B_extra_excluded": valid["A"] - valid["B"],
        "A_stores": len(stores["A"]), "B_stores": len(stores["B"]),
        "stores_lost_in_B": len(stores["A"] - stores["B"]),
    }

# ----------------------------- S02: customer reviews -----------------------------
S02_REQUIRED_KEYS = ["parent_asin", "rating", "text"]
TAG_RE = re.compile(r"<[^>]+>")

def clean_review_text(value):
    value = TAG_RE.sub(" ", value or "")
    value = html.unescape(value)
    return re.sub(r"\s+", " ", value).strip()

def review_is_valid(rec, config):
    """Config A: required keys present, raw text not blank.
    Config B: HTML tags removed first, text must not be blank afterwards."""
    if not isinstance(rec, dict) or not all(k in rec for k in S02_REQUIRED_KEYS):
        return False
    if rec["rating"] is None or not non_blank(rec["parent_asin"]) or not isinstance(rec["text"], str):
        return False
    if config == "B":
        return bool(clean_review_text(rec["text"]))
    return bool(rec["text"].strip())

def prepare_review_record(rec, config="B"):
    clean = config == "B"
    text = clean_review_text(rec["text"]) if clean else rec["text"].strip()
    title = clean_review_text(rec.get("title")) if clean else (rec.get("title") or "").strip()
    timestamp = rec.get("timestamp")
    review_date = None
    if isinstance(timestamp, (int, float)):
        review_date = datetime.fromtimestamp(timestamp / 1000, tz=timezone.utc).strftime("%Y-%m-%d")
    key = f"{rec['parent_asin']}|{timestamp}|{rec['text']}"
    return {
        "record_id": "rev-" + hashlib.sha1(key.encode("utf-8")).hexdigest()[:12],
        "source_id": "S02",
        "parent_asin": rec["parent_asin"],
        "asin": rec.get("asin"),
        "rating": rec["rating"],
        "title": title,
        "text": text,
        "helpful_vote": rec.get("helpful_vote"),
        "verified_purchase": rec.get("verified_purchase"),
        "review_date_utc": review_date,
    }

def compare_review_configs(src):
    total = 0
    valid = {"A": 0, "B": 0}
    chars = {"A": 0, "B": 0}
    with_tags = 0
    for rec in iter_jsonl(src):
        total += 1
        raw_text = rec.get("text") if isinstance(rec.get("text"), str) else ""
        if TAG_RE.search(raw_text):
            with_tags += 1
        for cfg in ("A", "B"):
            if review_is_valid(rec, cfg):
                valid[cfg] += 1
                chars[cfg] += len(clean_review_text(raw_text)) if cfg == "B" else len(raw_text.strip())
    return {
        "records_in": total,
        "A_valid": valid["A"], "B_valid": valid["B"],
        "reviews_with_html_tags": with_tags,
        "A_avg_chars": round(chars["A"] / max(valid["A"], 1), 1),
        "B_avg_chars": round(chars["B"] / max(valid["B"], 1), 1),
    }

def prepare_json_file(src, target, kind, config):
    """Write one prepared JSONL file from one JSON source. kind is 'meta' or 'reviews'."""
    is_valid = meta_is_valid if kind == "meta" else review_is_valid
    prepare = prepare_meta_record if kind == "meta" else prepare_review_record
    total = written = 0
    with Path(target).open("w", encoding="utf-8") as out:
        for rec in iter_jsonl(src):
            total += 1
            if not is_valid(rec, config):
                continue
            out.write(json.dumps(prepare(rec, config), ensure_ascii=False) + "\n")
            written += 1
    if written == 0:
        raise ValueError("No valid JSON records remain after validation.")
    return {"records_in": total, "records_out": written, "excluded": total - written}

# ----------------------------- S03-S05: PDF documents -----------------------------
CHUNK_FIXED = 500       # baseline from the lab
CHUNK_TARGET = 800      # selected: sentence-aware chunks, never across pages
CHUNK_MAX = 1000

PDF_RULES = {
    "S03": {"windows": None, "cut_regex": r"^references$"},
    "S04": {"windows": None, "cut_regex": r"^references$"},
    "S05": {
        "windows": [
            [r"^1\.\s+introduction$", r"^2\.\s+history of audio equalization$"],
            [r"^6\.\s+sound reproduction applications$", r"^7\.\s+audio content creation$"],
        ],
        "cut_regex": r"^references$",
    },
}

def extract_pages(pdf_path):
    reader = PdfReader(str(pdf_path))
    return [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]

def drop_repeated_lines(lines, n_pages):
    """Remove running headers and footers: lines (digits ignored) on at least half of the pages."""
    if n_pages < 4:
        return lines, 0
    norm = lambda s: re.sub(r"\d+", "#", s.strip())
    pages_with = defaultdict(set)
    for page, line in lines:
        key = norm(line)
        if key:
            pages_with[key].add(page)
    noise = {k for k, pages in pages_with.items() if len(pages) >= max(3, n_pages * 0.5)}
    kept = [(p, ln) for p, ln in lines if not norm(ln) or norm(ln) not in noise]
    return kept, len(lines) - len(kept)

def cut_references(lines, cut_regex):
    """Cut everything from the last line that is exactly 'References'."""
    if not cut_regex:
        return lines, False
    pattern = re.compile(cut_regex, re.I)
    cut_at = None
    for index, (_, line) in enumerate(lines):
        if pattern.match(line.strip()):
            cut_at = index
    if cut_at is None:
        print("WARNING: reference heading not found; references were not removed.")
        return lines, False
    return lines[:cut_at], True

def apply_windows(lines, windows):
    """Keep only the text between start and end headings. Fail loudly if a start heading is missing."""
    if not windows:
        return lines
    kept = []
    for start_re, end_re in windows:
        start, end = re.compile(start_re, re.I), re.compile(end_re, re.I)
        inside = found = False
        for page, line in lines:
            text = line.strip()
            if not inside and start.match(text):
                inside = found = True
            elif inside and end.match(text):
                break
            if inside:
                kept.append((page, line))
        if not found:
            raise ValueError(f"Start heading not found: {start_re}. Check the page text and adjust PDF_RULES.")
    return kept

def reflow(lines):
    """Join the lines of one page into running text and repair hyphenated line breaks."""
    text = ""
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if text.endswith("-") and not text.endswith(" -"):
            text = text[:-1] + line
        else:
            text = f"{text} {line}" if text else line
    return re.sub(r"\s+", " ", text).strip()

def build_page_texts(src, rule):
    pages = extract_pages(src)
    if not any(text.strip() for _, text in pages):
        raise ValueError(f"No extractable text in {Path(src).name}. Scanned PDFs may need OCR.")
    lines = [(page, line) for page, text in pages for line in text.splitlines()]
    lines, repeated = drop_repeated_lines(lines, len(pages))
    lines, references_cut = cut_references(lines, rule.get("cut_regex"))
    lines = apply_windows(lines, rule.get("windows"))
    by_page = {}
    for page, line in lines:
        by_page.setdefault(page, []).append(line)
    page_text = {page: reflow(ls) for page, ls in by_page.items()}
    page_text = {page: text for page, text in page_text.items() if text}
    if not page_text:
        raise ValueError(f"No text left after selection in {Path(src).name}.")
    notes = {
        "pages_total": len(pages),
        "pages_kept": len(page_text),
        "repeated_lines_removed": repeated,
        "references_cut": references_cut,
    }
    return page_text, notes

SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"\u201c(\[])")

def fixed_chunks(text, size=CHUNK_FIXED):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [text[i:i + size] for i in range(0, len(text), size)]

def page_sentence_chunks(page_text, target=CHUNK_TARGET, max_size=CHUNK_MAX):
    chunks, current = [], ""
    for sentence in SENTENCE_SPLIT.split(page_text):
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(sentence) > max_size:
            if current:
                chunks.append(current)
                current = ""
            chunks.extend(sentence[i:i + max_size] for i in range(0, len(sentence), max_size))
        elif current and len(current) + 1 + len(sentence) > max_size:
            chunks.append(current)
            current = sentence
        elif current and len(current) >= target:
            chunks.append(current)
            current = sentence
        else:
            current = f"{current} {sentence}" if current else sentence
    if current:
        chunks.append(current)
    return chunks

def chunk_quality(chunks):
    n = len(chunks)
    ends = sum(1 for c in chunks if re.search(r"[.!?][\"\u201d)\]]?$", c.strip()))
    starts = sum(1 for c in chunks if re.match(r"[A-Z0-9\"\u201c(\[]", c.strip()))
    return {
        "chunks": n,
        "avg_chars": round(sum(len(c) for c in chunks) / max(n, 1), 1),
        "pct_end_at_sentence": round(100 * ends / max(n, 1), 1),
        "pct_start_at_sentence": round(100 * starts / max(n, 1), 1),
    }

def pdf_records(source_id, src, page_text, target=CHUNK_TARGET, max_size=CHUNK_MAX):
    meta = SOURCE_META[Path(src).name]
    sha = source_hash(src)
    approach = f"page-aware-sentence-pack-target{target}-max{max_size}"
    records = []
    for page in sorted(page_text):
        for index, chunk in enumerate(page_sentence_chunks(page_text[page], target, max_size), 1):
            records.append({
                "record_id": f"{source_id}-p{page:03d}-{index:02d}",
                "source_id": source_id,
                "source": Path(src).name,
                "source_sha256": sha,
                "doi": meta.get("doi"),
                "title": meta["title"],
                "licence": meta.get("licence"),
                "page": page,
                "chunk_index": index,
                "text": chunk,
                "processing": approach,
            })
    return records

print("Cell 8 completed successfully. Project helpers are ready.")

Cell 8 completed successfully. Project helpers are ready.


## Part A — Inspect the sources (two representative records or pages per source)

In [19]:
# Cell 9 — A3 inspection: show real content, not only that the files exist
S01_FILE = select_source(relative_path=S01_NAME)
S02_FILE = select_source(relative_path=S02_NAME)

for label, path in (("S01 product metadata", S01_FILE), ("S02 customer reviews", S02_FILE)):
    rows = read_jsonl(path, limit=2000)
    print(f"=== {label}: {path.name} ===")
    print("Sample size read:", len(rows), "| keys in first record:", list(rows[0].keys()))
    for rec in rows[:2]:
        shown = {k: v for k, v in rec.items() if k not in ("images", "videos", "user_id")}
        print(json.dumps(shorten(shown), indent=2, ensure_ascii=False))
    print()

for source_id, name in PDF_NAMES.items():
    path = select_source(relative_path=name)
    pages = extract_pages(path)
    with_text = sum(1 for _, text in pages if len(text.strip()) > 100)
    print(f"=== {source_id}: {name} ===")
    print("Pages:", len(pages), "| pages with extractable text:", with_text)
    assert with_text > 0, "No extractable text. This PDF needs OCR."
    for page_no, text in pages[:2]:
        print(f"--- page {page_no} (first 250 characters) ---")
        print(repr(text.strip()[:250]))
    print()

=== S01 product metadata: s01_headphones_meta.jsonl ===
Sample size read: 2000 | keys in first record: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']
{
  "main_category": "All Electronics",
  "title": "MXditect EB166 Headphones with Mic Stereo Earphones for iOS Android Windows System, Black",
  "average_rating": 3.3,
  "rating_number": 69,
  "features": [
    "Equipped with two high quality drivers, create dynamic stereo sound and strong deep bass",
    "Tangle-free audio cord, helps you get rid of messy tangle effectively",
    "Include two in-line controllers with high fidelity microphone, allow hands-free operations",
    "...[1 more]"
  ],
  "description": [
    "Parameters: Package Size: 3.35 x 3.35 x 0.98 inches. Net Weight: 0.58 ounces. Rough Weight: 0.99 ounces. Color: Black. Cable Length: 120±5CM. Plug Diameter：3.5M...

## Part B — Prepare the data

### Pathway D — S01 product metadata

**Comparison on the same input.** Config A keeps every product with a non-blank `parent_asin` and `title`. Config B also requires at least one non-blank `features` or `description` text. The selected configuration is set in `CHOSEN_S01_CONFIG`.

In [20]:
# Cell 10 — PATHWAY D for S01 (product metadata): compare, export, validate, register lineage
S01_COMPARISON = compare_meta_configs(S01_FILE)
EVIDENCE["s01_comparison"] = S01_COMPARISON

print("--- S01 comparison (same input, two configurations) ---")
for key, value in S01_COMPARISON.items():
    print(f"{key}: {value}")

# Selected configuration: A keeps low-information products and flags them
# (has_features, has_description), so coverage by brand can still be measured later (RAI3).
CHOSEN_S01_CONFIG = "A"

prepared_meta_path = PROCESSED_DIR / "prepared_meta.jsonl"
S01_STATS = prepare_json_file(S01_FILE, prepared_meta_path, "meta", CHOSEN_S01_CONFIG)
EVIDENCE["s01_prepared"] = {"config": CHOSEN_S01_CONFIG, **S01_STATS}

# Read back and show a representative prepared record
exported = read_jsonl(prepared_meta_path, limit=3)
assert len(exported) > 0
print("\nPrepared records:", S01_STATS)
print("Representative prepared record:")
print(json.dumps(shorten(exported[0]), indent=2, ensure_ascii=False))

# B3 basic output contract
prepared_path = prepared_meta_path
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
print("Output:", prepared_path, "Size:", prepared_path.stat().st_size)
print("Validation: file exists and is non-empty; content inspected above.")

# B4.1 lineage registration
register_lineage(
    source=S01_FILE,
    output=prepared_meta_path,
    processing=(
        f"JSONL to prepared JSONL; config={CHOSEN_S01_CONFIG}; required_keys={S01_REQUIRED_KEYS}; "
        "keep text, rating, store and category fields; drop images/videos/bought_together/subtitle/author; "
        "parse details JSON string; price kept as 2023 snapshot; add has_* flags; "
        f"excluded_records={S01_STATS['excluded']}"
    ),
)

--- S01 comparison (same input, two configurations) ---
records_in: 56048
A_valid: 56043
B_valid: 44515
B_extra_excluded: 11528
A_stores: 10936
B_stores: 8648
stores_lost_in_B: 2288

Prepared records: {'records_in': 56048, 'records_out': 56043, 'excluded': 5}
Representative prepared record:
{
  "record_id": "meta-B06XJHRP5G",
  "source_id": "S01",
  "parent_asin": "B06XJHRP5G",
  "title": "MXditect EB166 Headphones with Mic Stereo Earphones for iOS Android Windows System, Black",
  "store": "MXditect",
  "categories": [
    "Electronics",
    "Headphones, Earbuds & Accessories",
    "Headphones & Earbuds",
    "...[1 more]"
  ],
  "subtype": "Earbud Headphones",
  "average_rating": 3.3,
  "rating_number": 69,
  "price_2023_snapshot": null,
  "features": [
    "Equipped with two high quality drivers, create dynamic stereo sound and strong deep bass",
    "Tangle-free audio cord, helps you get rid of messy tangle effectively",
    "Include two in-line controllers with high fidelity micro

### Pathway D — S02 customer reviews

**Comparison on the same input.** Config A keeps the review text as downloaded. Config B removes HTML tags (such as `<br />`) and decodes entities before checking that the text is not blank. The selected configuration is set in `CHOSEN_S02_CONFIG`.

In [21]:
# Cell 11 — PATHWAY D for S02 (customer reviews): compare, export, validate, register lineage
S02_COMPARISON = compare_review_configs(S02_FILE)
EVIDENCE["s02_comparison"] = S02_COMPARISON

print("--- S02 comparison (same input, two configurations) ---")
for key, value in S02_COMPARISON.items():
    print(f"{key}: {value}")

# Selected configuration: B removes HTML noise so retrieved passages are clean text.
CHOSEN_S02_CONFIG = "B"

prepared_reviews_path = PROCESSED_DIR / "prepared_reviews.jsonl"
S02_STATS = prepare_json_file(S02_FILE, prepared_reviews_path, "reviews", CHOSEN_S02_CONFIG)
EVIDENCE["s02_prepared"] = {"config": CHOSEN_S02_CONFIG, **S02_STATS}

exported = read_jsonl(prepared_reviews_path, limit=3)
assert len(exported) > 0
assert all("user_id" not in row for row in exported), "user_id must not appear in prepared data"
print("\nPrepared records:", S02_STATS)
print("Representative prepared record (shortened):")
print(json.dumps(shorten(exported[0]), indent=2, ensure_ascii=False))

# B3 basic output contract
prepared_path = prepared_reviews_path
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
print("Output:", prepared_path, "Size:", prepared_path.stat().st_size)
print("Validation: file exists and is non-empty; content inspected above.")

# B4.1 lineage registration
register_lineage(
    source=S02_FILE,
    output=prepared_reviews_path,
    processing=(
        f"JSONL to prepared JSONL; config={CHOSEN_S02_CONFIG}; required_keys={S02_REQUIRED_KEYS}; "
        "strip HTML tags and decode entities; drop user_id and images; add review_date_utc; "
        f"excluded_records={S02_STATS['excluded']}"
    ),
)

--- S02 comparison (same input, two configurations) ---
records_in: 39046
A_valid: 39033
B_valid: 39033
reviews_with_html_tags: 7529
A_avg_chars: 368.4
B_avg_chars: 361.4

Prepared records: {'records_in': 39046, 'records_out': 39033, 'excluded': 13}
Representative prepared record (shortened):
{
  "record_id": "rev-87be86659460",
  "source_id": "S02",
  "parent_asin": "B01G8JO5F2",
  "asin": "B01G8JO5F2",
  "rating": 5.0,
  "title": "Excellent!",
  "text": "I love these. They even come with a carry case and several sizes of ear bud inserts. Thank heaven! I get ear pain from most, but the smallest buds fit great. Th...[cut]",
  "helpful_vote": 0,
  "verified_purchase": true,
  "review_date_utc": "2018-04-07"
}
Output: /content/week3_project/data/processed/prepared_reviews.jsonl Size: 23906805
Validation: file exists and is non-empty; content inspected above.
Registered: s02_headphones_reviews.jsonl -> prepared_reviews.jsonl


### Pathway A — S03, S04, S05 research papers

**Comparison on the same input.** Config A is the lab baseline: all selected text joined and cut into fixed 500-character chunks (no page numbers). Config B keeps page numbers and packs whole sentences into chunks of about 800 characters (maximum 1,000), never across pages. Both start from the same cleaned text: running headers removed, references removed, and for S05 only Sections 1 and 6 kept.

In [22]:
# Cell 12 — PATHWAY A for S03-S05 (PDF papers): compare, export, validate, register lineage
all_pdf_records = []
pdf_sources = {}

for source_id, name in PDF_NAMES.items():
    src = select_source(relative_path=name)
    pdf_sources[source_id] = src
    page_text, notes = build_page_texts(src, PDF_RULES[source_id])

    joined = " ".join(page_text[p] for p in sorted(page_text))
    chunks_a = fixed_chunks(joined, CHUNK_FIXED)
    records_b = pdf_records(source_id, src, page_text)
    chunks_b = [r["text"] for r in records_b]

    quality_a, quality_b = chunk_quality(chunks_a), chunk_quality(chunks_b)
    EVIDENCE.setdefault("pdf", {})[source_id] = {"notes": notes, "A_fixed_500": quality_a, "B_page_sentence": quality_b}

    print(f"=== {source_id}: {name} ===")
    print("Selection notes:", notes)
    print("A fixed-500   :", quality_a, "| page numbers kept: no")
    print("B page-aware  :", quality_b, "| page numbers kept: yes")
    print("A sample:", repr(chunks_a[1][:160]) if len(chunks_a) > 1 else repr(chunks_a[0][:160]))
    print("B sample:", repr(chunks_b[1][:160]) if len(chunks_b) > 1 else repr(chunks_b[0][:160]))
    print()
    all_pdf_records.extend(records_b)

# Selected configuration: B (page-aware sentence packing)
approach = f"page-aware-sentence-pack-target{CHUNK_TARGET}-max{CHUNK_MAX}"
prepared_text_path = PROCESSED_DIR / "prepared_text.jsonl"
with prepared_text_path.open("w", encoding="utf-8") as out:
    for row in all_pdf_records:
        out.write(json.dumps(row, ensure_ascii=False) + "\n")

exported_records = read_jsonl(prepared_text_path)
assert len(exported_records) == len(all_pdf_records), "Exported record count does not match"
assert all(r["text"].strip() for r in exported_records), "One or more chunks are empty"
assert all(isinstance(r["page"], int) for r in exported_records), "Every chunk needs a page number"
EVIDENCE["pdf_prepared"] = {"approach": approach, "records": len(exported_records),
                            "per_source": dict(Counter(r["source_id"] for r in exported_records))}

print("Total chunks exported:", len(exported_records), dict(Counter(r["source_id"] for r in exported_records)))
print("First record:", json.dumps(shorten(exported_records[0]), indent=2, ensure_ascii=False))

# B3 basic output contract
prepared_path = prepared_text_path
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
print("Output:", prepared_path, "Size:", prepared_path.stat().st_size)
print("Validation: file exists and is non-empty; content inspected above.")

# B4.1 lineage registration, one entry per PDF
for source_id, src in pdf_sources.items():
    register_lineage(
        source=src,
        output=prepared_text_path,
        processing=f"{approach}; per-page text; running headers and references removed; "
                   f"rules={json.dumps(PDF_RULES[source_id])}",
    )

=== S03: s03_lee_2025_wireless_earbuds.pdf ===
Selection notes: {'pages_total': 11, 'pages_kept': 8, 'repeated_lines_removed': 30, 'references_cut': True}
A fixed-500   : {'chunks': 76, 'avg_chars': 493.7, 'pct_end_at_sentence': 2.6, 'pct_start_at_sentence': 7.9} | page numbers kept: no
B page-aware  : {'chunks': 48, 'avg_chars': 780.6, 'pct_end_at_sentence': 85.4, 'pct_start_at_sentence': 87.5} | page numbers kept: yes
A sample: 'Author: Chandrasekar Subramaniam, PhD Department of Business Information Systems and Operations Management University of North Carolina Charlotte 9201 Universit'
B sample: 'However, aspects including personal safety and fashion perspectives of wearable devices have not yet been adequately addressed in the literature. There have bee'

=== S04: s04_gutierrez_parera_2016_headphone_quality.pdf ===
Selection notes: {'pages_total': 18, 'pages_kept': 17, 'repeated_lines_removed': 17, 'references_cut': True}
A fixed-500   : {'chunks': 100, 'avg_chars': 499.5, 'pct_en

In [23]:
# Cell 13 — GENERATE THE COMMON LINEAGE MANIFEST (Cell 6A in the lab, with source details added)
assert lineage_entries, "No lineage entries registered. Register at least one selected pathway first."

lineage_records = []
for entry in lineage_entries.values():
    source, output, processing = entry["source"], entry["output"], entry["processing"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"
    meta = SOURCE_META[source.name]
    lineage_records.append({
        "source_id": meta["source_id"],
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": meta["version"],
        "access_rights": meta["access_rights"],
        "provenance_limitations": meta["limitations"],
        "status": "Prepared; quality review pending (Topic 4)",
    })

manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print("Source IDs:", [r["source_id"] for r in manifest["records"]])
print("Saved to:", BASE_DIR / "data" / "week3_manifest.json")

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 5
Source IDs: ['S01', 'S02', 'S03', 'S04', 'S05']
Saved to: /content/week3_project/data/week3_manifest.json


In [24]:
# Cell 14 — VERIFY THE COMMON LINEAGE MANIFEST (Cell 6B in the lab)
import json

manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

saved_records = saved_manifest.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."

for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared output checksum mismatch: {output}"

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 5
Manifest: /content/week3_project/data/week3_manifest.json


## Part C — Executable V1 data pipeline

In [25]:
# Cell 15 — REQUIRED: Common V1 pipeline utilities (Cell 7 in the lab)
# One change from the lab: a pathway may take a LIST of source files (S01 + S02, or S03 + S04 + S05).
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Registered pipeline runs, kept for the repeatability check
PIPELINE_RUNS = {}

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(f"Source folder not found: {source}")
    else:
        if not source.is_file():
            raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    # INGEST: identify the source files
    if isinstance(source, (list, tuple)):
        inputs = [Path(p) for p in source]
    elif kind == "images":
        inputs = sorted(p for p in Path(source).iterdir()
                        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})
    else:
        inputs = [Path(source)]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")

    # INSPECT: data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: selected transformation
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]

    # VALIDATE: prepared outputs exist, are not empty, and are fingerprinted
    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")
    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs],
    }

    # RECORD LINEAGE: pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": str(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "See data/week3_manifest.json and docs/week3_data_inventory.md",
        "access_rights": "Not verified for sharing: S01 and S02 have no stated licence. Check before sharing.",
    }
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", manifest_path)
    return record

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 15 ready.")
print("Run ONLY the Cell 16 (JSON) and Cell 17 (documents) pathways.")

Cell 15 ready.
Run ONLY the Cell 16 (JSON) and Cell 17 (documents) pathways.


In [26]:
# Cell 16 — PATHWAY D pipeline: S01 + S02 (JSON/JSONL records)
# Requires Cell 8 (project helpers) and Cell 15 (pipeline utilities).
import json

SOURCES_D = [resolve_source(S01_NAME), resolve_source(S02_NAME)]

# Same settings as Part B (Cells 10 and 11)
CONFIG_D = {
    "s01": {"config": CHOSEN_S01_CONFIG, "required_keys": S01_REQUIRED_KEYS},
    "s02": {"config": CHOSEN_S02_CONFIG, "required_keys": S02_REQUIRED_KEYS},
    "invalid_record_policy": "exclude",
}

def _route(path):
    if path.name == S01_NAME:
        return "s01", "meta", "prepared_meta.jsonl"
    if path.name == S02_NAME:
        return "s02", "reviews", "prepared_reviews.jsonl"
    raise ValueError(f"Unexpected JSON source: {path.name}")

def inspect_json(inputs, cfg):
    result = {}
    for src in inputs:
        key, kind, _ = _route(src)
        is_valid = meta_is_valid if kind == "meta" else review_is_valid
        total = valid = 0
        first_keys = None
        for rec in iter_jsonl(src):
            total += 1
            if first_keys is None and isinstance(rec, dict):
                first_keys = list(rec.keys())
            if is_valid(rec, cfg[key]["config"]):
                valid += 1
        if valid == 0:
            raise ValueError(f"No valid records in {src.name}")
        result[src.name] = {"record_count": total, "valid_records": valid,
                            "excluded_records": total - valid,
                            "required_keys": cfg[key]["required_keys"], "first_keys": first_keys}
    return result

def prepare_json(inputs, out, cfg):
    if cfg["invalid_record_policy"] != "exclude":
        raise ValueError("This V1 pipeline supports only the 'exclude' policy.")
    outputs, mapping, notes = [], [], []
    for src in inputs:
        key, kind, out_name = _route(src)
        target = out / out_name
        stats = prepare_json_file(src, target, kind, cfg[key]["config"])
        exported = sum(1 for _ in iter_jsonl(target))          # read-back check
        if exported != stats["records_out"]:
            raise ValueError(f"Exported record count does not match for {out_name}")
        outputs.append(target)
        mapping.append({"source": str(src), "output": str(target), "records_out": stats["records_out"]})
        notes.append(f"{out_name}: config={cfg[key]['config']}, out={stats['records_out']}, excluded={stats['excluded']}")
    return outputs, mapping, "JSONL to prepared JSONL; " + "; ".join(notes)

run_D = register_and_run("json", SOURCES_D, inspect_json, prepare_json, CONFIG_D)
print("Cell 16 completed successfully.")
print("Inspection:", json.dumps(run_D["inspection"], indent=2))

SUCCESS: json | Inputs: 2 | Outputs: 2
Manifest: /content/week3_project/data/pipeline_v1/json/pipeline_manifest.json
Cell 16 completed successfully.
Inspection: {
  "s01_headphones_meta.jsonl": {
    "record_count": 56048,
    "valid_records": 56043,
    "excluded_records": 5,
    "required_keys": [
      "parent_asin",
      "title"
    ],
    "first_keys": [
      "main_category",
      "title",
      "average_rating",
      "rating_number",
      "features",
      "description",
      "price",
      "images",
      "videos",
      "store",
      "categories",
      "details",
      "parent_asin",
      "bought_together",
      "subtitle",
      "author"
    ]
  },
  "s02_headphones_reviews.jsonl": {
    "record_count": 39046,
    "valid_records": 39033,
    "excluded_records": 13,
    "required_keys": [
      "parent_asin",
      "rating",
      "text"
    ],
    "first_keys": [
      "rating",
      "title",
      "text",
      "images",
      "asin",
      "parent_asin",
      "us

In [27]:
# Cell 17 — PATHWAY A pipeline: S03 + S04 + S05 (PDF papers)
# Requires Cell 8 (project helpers) and Cell 15 (pipeline utilities).
import json

SOURCES_A = [resolve_source(name) for name in PDF_NAMES.values()]

# Same settings as Part B (Cell 12)
CONFIG_A = {
    "method": "page-aware-sentence-pack",
    "target_chars": CHUNK_TARGET,
    "max_chars": CHUNK_MAX,
    "rules": PDF_RULES,
}

def _source_id(path):
    for source_id, name in PDF_NAMES.items():
        if path.name == name:
            return source_id
    raise ValueError(f"Unexpected PDF source: {path.name}")

def inspect_documents(inputs, cfg):
    result = {}
    for src in inputs:
        pages = extract_pages(src)
        with_text = sum(1 for _, text in pages if len(text.strip()) > 100)
        if with_text == 0:
            raise ValueError(f"No extractable text in {src.name}. Scanned PDFs may need OCR.")
        result[src.name] = {"pages": len(pages), "pages_with_text": with_text,
                            "characters": sum(len(text) for _, text in pages)}
    return result

def prepare_documents(inputs, out, cfg):
    records, mapping = [], []
    for src in inputs:
        source_id = _source_id(src)
        page_text, notes = build_page_texts(src, cfg["rules"][source_id])
        rows = pdf_records(source_id, src, page_text, cfg["target_chars"], cfg["max_chars"])
        if not rows:
            raise ValueError(f"No text chunks produced for {src.name}")
        records.extend(rows)
        mapping.append({"source": str(src), "output": str(out / "prepared_text.jsonl"),
                        "chunks": len(rows), "selection": notes})
    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for row in records:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    method = (f"{cfg['method']} target={cfg['target_chars']} max={cfg['max_chars']}; "
              "per-page text, running headers and references removed")
    return [target], mapping, method

run_A = register_and_run("documents", SOURCES_A, inspect_documents, prepare_documents, CONFIG_A)
print("Cell 17 completed successfully.")
print("Inspection:", json.dumps(run_A["inspection"], indent=2))

SUCCESS: documents | Inputs: 3 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Cell 17 completed successfully.
Inspection: {
  "s03_lee_2025_wireless_earbuds.pdf": {
    "pages": 11,
    "pages_with_text": 11,
    "characters": 46276
  },
  "s04_gutierrez_parera_2016_headphone_quality.pdf": {
    "pages": 18,
    "pages_with_text": 18,
    "characters": 55377
  },
  "s05_valimaki_reiss_2016_equalization.pdf": {
    "pages": 46,
    "pages_with_text": 46,
    "characters": 160258
  }
}


In [28]:
# Cell 18 — REQUIRED: Repeatability and provenance verification (Cell 9 in the lab)
assert PIPELINE_RUNS, "No pipeline runs registered. Run at least one selected pathway first."

print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()                      # run the same pipeline again

    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], (
        f"Source-to-output mapping changed for {kind}"
    )
    print(f"PASS: {kind} - source fingerprints, prepared outputs and mappings match")

print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: json | Inputs: 2 | Outputs: 2
Manifest: /content/week3_project/data/pipeline_v1/json/pipeline_manifest.json
PASS: json - source fingerprints, prepared outputs and mappings match
SUCCESS: documents | Inputs: 3 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents - source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


## Package the evidence for GitHub

In [29]:
# Cell 19 — Package the run evidence and the files that are safe to commit
# S01 and S02 have no stated licence, so only small samples and manifests are committed for them.
import shutil

COMMIT_DIR = Path("/content/week3_commit")
if COMMIT_DIR.exists():
    shutil.rmtree(COMMIT_DIR)
for sub in ("data/processed", "data/samples", "data/pipeline_v1/json", "data/pipeline_v1/documents", "docs"):
    (COMMIT_DIR / sub).mkdir(parents=True, exist_ok=True)

def copy(src, dst):
    shutil.copy2(src, COMMIT_DIR / dst)

copy(BASE_DIR / "data" / "week3_manifest.json", "data/week3_manifest.json")
copy(PROCESSED_DIR / "prepared_text.jsonl", "data/processed/prepared_text.jsonl")          # S03-S05 (open licence)
copy(PIPE_DIR / "documents" / "prepared_text.jsonl", "data/pipeline_v1/documents/prepared_text.jsonl")
copy(PIPE_DIR / "documents" / "pipeline_manifest.json", "data/pipeline_v1/documents/pipeline_manifest.json")
copy(PIPE_DIR / "json" / "pipeline_manifest.json", "data/pipeline_v1/json/pipeline_manifest.json")  # manifest only

# Small samples of the prepared S01 and S02 data (first 20 records)
for name in ("prepared_meta", "prepared_reviews"):
    rows = read_jsonl(PROCESSED_DIR / f"{name}.jsonl", limit=20)
    with (COMMIT_DIR / "data" / "samples" / f"{name}_sample.jsonl").open("w", encoding="utf-8") as out:
        for row in rows:
            out.write(json.dumps(row, ensure_ascii=False) + "\n")

# Evidence summary for docs/week3_pipeline_decisions.md
def evidence_markdown():
    s1, s2 = EVIDENCE["s01_comparison"], EVIDENCE["s02_comparison"]
    lines = ["# Week 3 run evidence", "",
             f"Generated (UTC): {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')}", "",
             "## S01 product metadata (Config A vs Config B)", "",
             "| Measure | Config A | Config B |", "|---|---|---|",
             f"| Records in the S01 extract | {s1['records_in']} | {s1['records_in']} |",
             f"| Valid records kept | {s1['A_valid']} | {s1['B_valid']} |",
             f"| Extra records excluded by B | 0 | {s1['B_extra_excluded']} |",
             f"| Distinct stores (brands) kept | {s1['A_stores']} | {s1['B_stores']} |",
             f"| Stores lost in B | 0 | {s1['stores_lost_in_B']} |", "",
             f"Selected: Config {EVIDENCE['s01_prepared']['config']}; prepared records: {EVIDENCE['s01_prepared']['records_out']}.", "",
             "## S02 customer reviews (Config A vs Config B)", "",
             "| Measure | Config A | Config B |", "|---|---|---|",
             f"| Records in the S02 extract | {s2['records_in']} | {s2['records_in']} |",
             f"| Valid records kept | {s2['A_valid']} | {s2['B_valid']} |",
             f"| Reviews containing HTML tags | {s2['reviews_with_html_tags']} | 0 after cleaning |",
             f"| Average characters per review | {s2['A_avg_chars']} | {s2['B_avg_chars']} |", "",
             f"Selected: Config {EVIDENCE['s02_prepared']['config']}; prepared records: {EVIDENCE['s02_prepared']['records_out']}.", "",
             "## S03 to S05 chunking (fixed 500 vs page-aware sentence packing)", "",
             "| Source | Pages kept / total | A chunks | B chunks | A ends at sentence % | B ends at sentence % | A starts at sentence % | B starts at sentence % |",
             "|---|---|---|---|---|---|---|---|"]
    for source_id, info in EVIDENCE["pdf"].items():
        a, b, n = info["A_fixed_500"], info["B_page_sentence"], info["notes"]
        lines.append(f"| {source_id} | {n['pages_kept']} / {n['pages_total']} | {a['chunks']} | {b['chunks']} | "
                     f"{a['pct_end_at_sentence']} | {b['pct_end_at_sentence']} | "
                     f"{a['pct_start_at_sentence']} | {b['pct_start_at_sentence']} |")
    lines += ["", f"Selected: B ({EVIDENCE['pdf_prepared']['approach']}); chunks exported: "
              f"{EVIDENCE['pdf_prepared']['records']} {EVIDENCE['pdf_prepared']['per_source']}.", "",
              "## Pipeline runs (Cell 18 repeatability: PASS)", ""]
    for kind, entry in PIPELINE_RUNS.items():
        first = entry["first"]
        lines.append(f"- **{kind}**: outputs {[Path(o).name for o in first['outputs']]}, "
                     f"output bytes {first['validation']['output_bytes']}, "
                     f"output SHA-256 {[h[:12] for h in first['validation']['output_sha256']]}")
    return "\n".join(lines) + "\n"

evidence_text = evidence_markdown()
(COMMIT_DIR / "docs" / "week3_run_evidence.md").write_text(evidence_text, encoding="utf-8")
print(evidence_text)

# Zip for download
zip_path = shutil.make_archive("/content/week3_commit", "zip", COMMIT_DIR)
print("Commit bundle:", zip_path)
for p in sorted(COMMIT_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(COMMIT_DIR), p.stat().st_size, "bytes")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab; download the zip manually.")

# Week 3 run evidence

Generated (UTC): 2026-11-04 08:10

## S01 product metadata (Config A vs Config B)

| Measure | Config A | Config B |
|---|---|---|
| Records in the S01 extract | 56048 | 56048 |
| Valid records kept | 56043 | 44515 |
| Extra records excluded by B | 0 | 11528 |
| Distinct stores (brands) kept | 10936 | 8648 |
| Stores lost in B | 0 | 2288 |

Selected: Config A; prepared records: 56043.

## S02 customer reviews (Config A vs Config B)

| Measure | Config A | Config B |
|---|---|---|
| Records in the S02 extract | 39046 | 39046 |
| Valid records kept | 39033 | 39033 |
| Reviews containing HTML tags | 7529 | 0 after cleaning |
| Average characters per review | 368.4 | 361.4 |

Selected: Config B; prepared records: 39033.

## S03 to S05 chunking (fixed 500 vs page-aware sentence packing)

| Source | Pages kept / total | A chunks | B chunks | A ends at sentence % | B ends at sentence % | A starts at sentence % | B starts at sentence % |
|---|---|---|---|---|---|---|---|

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>